# Climate State Finder

This notebook identifies **climate states** — sustained periods where a climate variable is anomalously high or low relative to a baseline — across all available WRF and LOCA2 simulations for a user-defined region and warming level.

**Methodology overview:**
1. Retrieve monthly climate data for a baseline and future Global Warming Level (GWL) for a spatial region of interest
2. Calculate anomalies relative to the baseline monthly climatology
3. Apply a rolling median over a user-defined duration window to smooth short-term variability
4. Define "high" and "low" climate state thresholds using the 75th and 25th percentiles of the baseline anomaly distribution
5. Flag time windows where the future anomaly consistently exceeds these thresholds for the full duration

**Inputs required:**
- Spatial domain (custom shapefile or built-in boundary)
- One variable: `"air temperature"`, `"precipitation"`, or `"relative humidity"`
- Baseline and future GWL

**Outputs:**
- A combined WRF + LOCA2 DataFrame with climate state labels per simulation and time step, saved as a `.parquet` file for downstream use (e.g., event finder)
- A facet plot showing anomaly timeseries with highlighted climate state windows

**Expected runtime:** ~10–12 minutes (dominated by LOCA2 data loading, ~7.5 min)

In [ ]:
import climakitae as ck
import geopandas as gpd
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from dask.diagnostics import ProgressBar

## 1. Make Selections

Use the ClimateData object to subset our data catalog. Here, we'll make selections and see available options for variables, spatial domain, and GWLs.

In [ ]:
cd = ck.ClimateData(verbosity=-1)

### 1a. Select a temporal resolution
The temporal resolution selection will impact which downscaling methods and variables are available.

In [ ]:
# Show temporal resolution (AKA "table id") options
# cd.show_table_id_options()

In [ ]:
table_id = "mon"  # Options are "mon", "day", "1hr"
cd.table_id(table_id)

### 1b. Select spatial resolution
Depending on your selections above, your spatial resolution options may change. Use `cd.show_grid_label_options()` to see the available spatial resolutions for your variable and dataset selections. A smaller resolution (e.g. 3 km) indicates a finer grid (e.g. high resolution) versus a coarser grid (e.g. 45 km). Note that you'll need to use the CMIP6 grid label wording in the code: `d01` (45 km), `d02` ( 9 km), or `d03` ( 3 km). 

In [ ]:
# Show spatial resolution ("grid label") options
# cd.show_grid_label_options()

In [ ]:
grid_label = "d03"  # 3 km
cd.grid_label(grid_label)

### 1c. Select a baseline and future GWL
The baseline Global Warming Level (GWL) is the historical warming level used as a reference for comparison. We suggest using 0.8 °C. Available warming levels are 0.8, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0, and 4.0 °C, where higher values represent a warmer future climate.

In [ ]:
# Set baseline and future global warming level as a float
baseline_gwl = 0.8
future_gwl = 2.0

### 1d. Select variable and units 
This notebook supports three variables: `"precipitation"`, `"relative humidity"`, or `"air temperature"`.

In [ ]:
variable = "precipitation"

Next, the plain-language variable name is mapped to the appropriate variable id for each dataset. The units are set here as well-- feel free to change them to your desired units (unit options are in the code comments). 

**NOTE**: LOCA2 only provides minimum and maximum values for air temperature and relative humidity. If these variables are desired, the notebook will automatically retrieve **both variables** and average them. Because of the additional processing for these LOCA2 variables, the `LOCA2_var` is a `list` rather than a `str`. 

In [ ]:
if variable == "precipitation":
    WRF_var = "prec"
    WRF_units = "inches"  # options: "mm", "inches"
    LOCA2_var = ["pr"]
    LOCA2_units = "inches"  # options: "kg m-2 s-1", "mm", "inches"
elif variable == "relative humidity":
    WRF_var = "rh"
    WRF_units = "%[0 to 100]"  # options: "%[0 to 100]"
    LOCA2_var = ["hursmax", "hursmin"]
    LOCA2_units = "percent"  # options: "percent"  (equivalent to %[0 to 100])
elif variable == "air temperature":
    WRF_var = "t2"
    WRF_units = "degF"  # options: "K", "degC", "degF"
    LOCA2_var = ["tasmax", "tasmin"]
    LOCA2_units = "degF"  # options: "K", "degC", "degF"

### 1e. Select aggregation method
Select a method to aggregate spatially (across the region of interest) and temporally (across time to a monthly timescale). The temporal aggregation step will only applied if you chose daily or hourly data, since monthly data is by definition already aggregated. 

In [ ]:
# options are mean, median, sum
temporal_aggregation = "mean"
spatial_aggregation = "mean"

### 1f. Select climate state duration
How often should the climate state be observed to be considered a 'hit', in units of months. 

In [ ]:
duration = 24  # months

### 1g. Validate selections 
Ensure the inputs are valid before proceeding to data retrieval 

In [ ]:
# Check GWL
valid_gwls = [0.8, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0, 4.0]
invalid = [gwl for gwl in [baseline_gwl, future_gwl] if gwl not in valid_gwls]
if invalid:
    raise ValueError(f"Invalid GWL value(s) {invalid}. Must be one of: {valid_gwls}")

# Check variable
valid_variables = ["precipitation", "relative humidity", "air temperature"]
if variable not in valid_variables:
    raise ValueError(
        f"Invalid climate_variable '{variable}'. Must be one of: {valid_variables}"
    )

# Check aggregation
valid_aggs = ["mean", "median", "sum"]
invalid = [
    agg for agg in [spatial_aggregation, temporal_aggregation] if agg not in valid_aggs
]
if invalid:
    raise ValueError(
        f"Invalid aggregation value(s) {invalid}. Must be one of: {valid_aggs}"
    )

## 2. Select Spatial Domain
A default shapefile is provided so you can explore the notebook workflow without additional setup. The example uses the Brawley hydrologic area in Imperial County, California, which makes up part of the Colorado River basin within the Salton Sea HUC-8 watershed.

To use your own study area, replace the shapefile path in the code below with the path to your custom shapefile.

In [ ]:
# Region of Interest shapefile
shapefile_filepath = "https://geo.sandag.org/server/rest/directories/downloads/Hydrological_Basins_shapefile.zip"
spatial_domain_name = "Brawley"
filter_field = "haname"  # column to filter domain name on

# Read in data and filter for specific ROI
shapefile_all = gpd.read_file(shapefile_filepath)
roi = shapefile_all[shapefile_all[filter_field] == spatial_domain_name]

# Generate quick plot
roi.plot();

## 3. Retrieve Data

### 3a. Retrieve WRF data 

In [ ]:
wrf_data = (
    cd.catalog("cadcat")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id(table_id)
    .grid_label(grid_label)
    .variable(WRF_var)
    .processes(
        {
            "warming_level": {
                "warming_levels": [baseline_gwl, future_gwl],
                "add_dummy_time": True,
            },
            "clip": roi,
            "convert_units": WRF_units,
            "metric_calc": {
                "metric": spatial_aggregation,
                "dim": ["x", "y"],  # Aggregation across spatial dimensions
            },
        }
    )
).get()

### 3b. Retrieve LOCA2 data 

In [ ]:
loca2_data = [
    (
        cd.catalog("cadcat")
        .activity_id("LOCA2")
        .institution_id("UCSD")
        .table_id(table_id)
        .grid_label(grid_label)
        .variable(var)
        .processes(
            {
                "warming_level": {
                    "warming_levels": [baseline_gwl, future_gwl],
                    "add_dummy_time": True,
                },
                "clip": roi,
                "convert_units": LOCA2_units,
                "metric_calc": {
                    "metric": spatial_aggregation,
                    "dim": ["lat", "lon"],
                },
            }
        )
    ).get()
    for var in LOCA2_var
]

### 3c. Preprocessing 
Next, we will perform some simple data preprocessing to ensure the WRF and LOCA2 data are in the proper format for the next operations. 

In [ ]:
# Convert Dataset --> DataArray object
wrf_data = wrf_data[WRF_var]
loca2_data = [loca2_data[i][var] for i, var in zip([0, 1], LOCA2_var)]

# LOCA2 preprocessing if two variables were retrieved
if len(loca2_data) > 1:
    # For air temperature and relative humidity, average the min and max values
    loca2_data = (loca2_data[0] + loca2_data[1]) / 2
else:
    # Otherwise, only one dataset is retrieved, and it's a singeton list
    loca2_data = loca2_data[0]

# Assign both datasets the same name
wrf_data.name = variable
loca2_data.name = variable

### 3d. Perform monthly aggregation
For daily or hourly data, aggregate to monthly resolution using the selected method. This step is skipped if monthly data was selected above.

In [ ]:
# create monthly aggegations
if table_id != "mon":
    wrf_data = eval(f"wrf_data.resample(time = '1M').{temporal_aggregation}()")
    loca2_data = eval(f"loca2_data.resample(time = '1M').{temporal_aggregation}()")
    print(f"Data was aggregated monthly using a {temporal_aggregation}")

### 3e. Read data into memory
Dask will print a progress bar as each dataset is loaded into memory.

In [ ]:
with ProgressBar():
    wrf_data.load()
    loca2_data.load()

## 4. Calculate Anomalies

Anomalies measure how much the future climate deviates from the baseline. For each simulation, we subtract the monthly climatological median (computed over the baseline GWL period) from both the future and baseline GWL data. This removes the seasonal cycle, isolating the anomalous signal. A rolling median is then applied to highlight sustained departures.

### 4a. Calculate monthly climatology for baseline GWL

Compute the median value for each calendar month across all simulations at the baseline GWL. This serves as the reference climatology against which anomalies are measured.

In [ ]:
# calculate monthly mean for each simulation for the baseline gwl
loca2_clim_mean = (
    loca2_data.drop_duplicates(dim="warming_level")
    .sel(warming_level=float(baseline_gwl))
    .groupby("time.month")
    .median()
)
wrf_clim_mean = (
    wrf_data.drop_duplicates(dim="warming_level")
    .sel(warming_level=float(baseline_gwl))
    .groupby("time.month")
    .median()
)

### 4b. Calculate anomalies relative to baseline climatology

Subtract the monthly climatology from both the future and baseline GWL data to produce anomaly timeseries for each simulation.

In [ ]:
# subtract the monthly climatology from the future gwl data to create an anomaly
loca2_anom = (
    loca2_data.drop_duplicates(dim="warming_level")
    .sel(warming_level=float(future_gwl))
    .groupby("time.month")
    - loca2_clim_mean
)
wrf_anom = (
    wrf_data.drop_duplicates(dim="warming_level")
    .sel(warming_level=float(future_gwl))
    .groupby("time.month")
    - wrf_clim_mean
)

In [ ]:
# calculate historical anomaly
hist_loca2_anom = (
    loca2_data.drop_duplicates(dim="warming_level")
    .sel(warming_level=float(baseline_gwl))
    .groupby("time.month")
    - loca2_clim_mean
)
hist_wrf_anom = (
    wrf_data.drop_duplicates(dim="warming_level")
    .sel(warming_level=float(baseline_gwl))
    .groupby("time.month")
    - wrf_clim_mean
)

### 4c. Apply rolling median

Smooth the anomaly timeseries using a rolling median over the `duration` window (in months). This filters out short-term variability and highlights sustained anomalies.

In [ ]:
loca2_anom = loca2_anom.rolling(time=duration).median()
wrf_anom = wrf_anom.rolling(time=duration).median()

In [ ]:
hist_loca2_anom = hist_loca2_anom.rolling(time=duration).median()
hist_wrf_anom = hist_wrf_anom.rolling(time=duration).median()

## 5. Identify Climate States

A climate state "hit" is defined as a sustained period where the future anomaly consistently exceeds a threshold derived from the baseline distribution. Thresholds are set at the 75th percentile (high state) and 25th percentile (low state) of the baseline rolling anomaly. A time window is flagged as a hit only if the anomaly exceeds the threshold for every month in the full `duration` window.

In [ ]:
# Create an upper and lower threshold
loca2_high = hist_loca2_anom.quantile(q=0.75, dim="time")
loca2_low = hist_loca2_anom.quantile(q=0.25, dim="time")
wrf_high = hist_wrf_anom.quantile(q=0.75, dim="time")
wrf_low = hist_wrf_anom.quantile(q=0.25, dim="time")

In [ ]:
# Add data to data array
wrf_anom = wrf_anom.assign_coords(climate_state_high=("sim", wrf_high.values))
wrf_anom = wrf_anom.assign_coords(climate_state_low=("sim", wrf_low.values))
loca2_anom = loca2_anom.assign_coords(climate_state_high=("sim", loca2_high.values))
loca2_anom = loca2_anom.assign_coords(climate_state_low=("sim", loca2_low.values))

Next we will search the data for the climate states defined above

In [ ]:
def compute_climate_hits(anom, high_thresh, low_thresh, duration):
    """
    Identify climate state 'hits' for each simulation and time step.

    Parameters
    ----------
    anom : xr.DataArray
        Anomaly data with dimensions (sim, time).
    high_thresh : xr.DataArray
        Upper threshold per simulation.
    low_thresh : xr.DataArray
        Lower threshold per simulation.
    duration : int
        Number of months required to constitute a hit.

    Returns
    -------
    np.ndarray
        Array of shape (sim, time) with 1 (high), -1 (low), or 0 (no hit).
    """
    climate_hit = np.zeros(anom.values.shape)
    n_times = len(anom["time"])

    for t in range(n_times):
        # get time indices for this window, clipped to array bounds
        time_idx = [i for i in range(t, t + duration) if i < n_times]
        anom_window = anom.isel(time=time_idx)

        # count how many time steps exceed each threshold per simulation
        high_hit = ((anom_window > high_thresh).sum(dim="time") >= duration).values
        low_hit = ((anom_window < low_thresh).sum(dim="time") >= duration).values

        # mark hits: 1 for high state, -1 for low state
        climate_hit[np.ix_(high_hit, time_idx)] = 1
        climate_hit[np.ix_(low_hit, time_idx)] = -1

    return climate_hit


wrf_climate_hit = compute_climate_hits(wrf_anom, wrf_high, wrf_low, duration)
loca2_climate_hit = compute_climate_hits(loca2_anom, loca2_high, loca2_low, duration)

In [ ]:
wrf_anom = wrf_anom.assign_coords(climate_state_hit=(("sim", "time"), wrf_climate_hit))
loca2_anom = loca2_anom.assign_coords(
    climate_state_hit=(("sim", "time"), loca2_climate_hit)
)

## 6. Prepare for Plotting

Convert the xarray DataArrays to a single pandas DataFrame. Each row represents one simulation–timestep pair and includes the anomaly value, climate state hit label, and metadata (model, realization, downscaling method).

In [ ]:
# Add model and realization metadata as coordinates
lsims = loca2_anom.sim.values.tolist()
wsims = wrf_anom.sim.values.tolist()

loca2_anom = loca2_anom.assign_coords(
    models=("sim", [s.split("_")[1] for s in lsims]),
    realization=("sim", [s.split("_")[2] + "_" + s.split("_")[3] for s in lsims]),
)
wrf_anom = wrf_anom.assign_coords(
    models=("sim", [s.split("_")[1] for s in wsims]),
    realization=("sim", [s.split("_")[2] + "_" + s.split("_")[3] for s in wsims]),
)

In [ ]:
# Convert to DataFrames, label downscaling method, and combine
loca2_df = loca2_anom.to_dataframe().reset_index()
wrf_df = wrf_anom.to_dataframe().reset_index()
loca2_df["downscaling"] = "loca2"
wrf_df["downscaling"] = "wrf"
final_df = pd.concat([loca2_df, wrf_df], ignore_index=True)

In [ ]:
final_df.head()

## 7. Visualize Results

Each panel shows the rolling anomaly timeseries for one simulation. **Maroon** shading highlights high climate state windows (anomaly exceeds the 75th percentile threshold for the full duration); **blue** shading highlights low climate state windows (anomaly below the 25th percentile).

In [ ]:
from matplotlib.patches import Patch


def annotate(data, **kws):
    ax = plt.gca()
    ax.axhline(0, ls="--", color="black", linewidth=0.8)
    ax.fill_between(
        data.time,
        data[variable],
        where=data.climate_state_hit == 1,
        facecolor="maroon",
        alpha=0.5,
    )
    ax.fill_between(
        data.time,
        data[variable],
        where=data.climate_state_hit == -1,
        facecolor="darkblue",
        alpha=0.5,
    )


g1 = sns.relplot(
    data=final_df,
    x="time",
    y=variable,
    col="sim",
    color="black",
    col_wrap=4,
    kind="line",
    height=5,
    aspect=1,
    facet_kws=dict(sharex=False),
)
g1.map_dataframe(annotate)
g1.set_axis_labels("Time", f"{variable.title()} anomaly ({WRF_units})")
g1.figure.suptitle(
    f"{variable.title()} anomaly at {future_gwl}°C GWL vs. {baseline_gwl}°C baseline  |  "
    f"{duration}-month rolling median  |  {spatial_aggregation} over {spatial_domain_name}",
    y=1.02,
)
legend_patches = [
    Patch(
        facecolor="maroon",
        alpha=0.5,
        label=f"High state (>75th pct, {duration}-month sustained)",
    ),
    Patch(
        facecolor="darkblue",
        alpha=0.5,
        label=f"Low state (<25th pct, {duration}-month sustained)",
    ),
]
g1.figure.legend(
    handles=legend_patches, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.02)
)
plt.show()

In [ ]:
# Save the figure to a file
g1.savefig(
    f"climate_state_finder_{variable}_under_{future_gwl}gwl_with_{duration}{table_id}duration.jpeg".replace(
        " ", "_"
    )
)

In [ ]:
# save climate state data (to be loaded into event finder)
final_df.to_parquet(
    f"climate_state_{variable}_under_{future_gwl}gwl_with_{duration}{table_id}duration".replace(
        " ", "_"
    )
)